# QM9 HOMO-LUMO Gap Prediction Challenge

**Goal:** Predict the HOMO-LUMO gap (in eV) of small molecules using an MLP.

The baseline uses only atom counts as features.  
Can you do better by using more of the available molecular information?

In [ ]:
! pip install -q flax jax optax torch-geometric

In [ ]:
import flax.linen as nn
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from torch_geometric.datasets import QM9

## 1. Explore the Data

QM9 contains ~130k small organic molecules. Each molecule is stored as a **graph**:

```
       H           Atoms (nodes): positions + element type
       |           Bonds (edges): which atoms are connected
   H - C - O - H
       |
       H
```

For our MLP, we need to convert each graph into a **fixed-size feature vector**. What information should we extract?

In [ ]:
# Downloads automatically on first run (~300 MB)
dataset = QM9(root="./data/qm9")
print(f"Dataset size: {len(dataset)} molecules")

### What's in each molecule?

| Field | What it is | Shape | Example use |
|-------|-----------|-------|-------------|
| `d.z` | Atomic numbers | `(n_atoms,)` | Count carbons: `(d.z == 6).sum()` |
| `d.pos` | 3D coordinates (Å) | `(n_atoms, 3)` | Molecular radius, distances |
| `d.edge_index` | Bond connectivity | `(2, n_edges)` | Which atoms are bonded |
| `d.edge_attr` | Bond types (one-hot) | `(n_edges, 4)` | Count double bonds |

Element codes: H=1, C=6, N=7, O=8, F=9

In [ ]:
# Explore the raw data
d = dataset[0]

print("Atomic numbers (d.z):", d.z.tolist())
print("3D positions (d.pos):")
print(d.pos.numpy().round(2))

## 2. Feature Extraction

Now let's extract features from molecules. The baseline just counts atoms.

**Your task:** Modify `featurize()` to extract better features!

In [ ]:
def featurize(d):
    """Convert molecule graph to fixed-size feature vector.

    Baseline: just count atoms of each element type.
    Can you do better?
    """
    z = d.z.numpy()

    # Baseline: atom counts [H, C, N, O, F]
    features = [
        (z == 1).sum(),   # Hydrogen
        (z == 6).sum(),   # Carbon
        (z == 7).sum(),   # Nitrogen
        (z == 8).sum(),   # Oxygen
        (z == 9).sum(),   # Fluorine
    ]

    # Add more features, e.g. mean/max coordinates, ...

    return np.array(features, dtype=np.float32)

In [ ]:
# Test your featurize function on a single molecule
d = dataset[0]
features = featurize(d)
print(f"Feature vector: {features}")
print(f"Number of features: {len(features)}")

## 3. Prepare Data for Training

Once you're happy with your features, run this to prepare the full dataset.

In [ ]:
def load_data(n_samples=32000):
    """Load QM9 and extract features."""
    X, y = [], []
    for i in range(min(n_samples, len(dataset))):
        X.append(featurize(dataset[i]))
        y.append(dataset[i].y[0, 4].item())  # HOMO-LUMO gap

    X, y = np.array(X), np.array(y, dtype=np.float32)
    print(f"Feature dim: {X.shape[1]}")

    # Normalize
    X = (X - X.mean(0)) / (X.std(0) + 1e-8)
    y_mean, y_std = y.mean(), y.std()
    y = (y - y_mean) / y_std

    # Shuffle and split
    perm = np.random.default_rng(42).permutation(len(X))
    X, y = X[perm], y[perm]
    n = int(0.8 * len(X))
    return X[:n], y[:n], X[n:], y[n:], y_std

In [ ]:
print("Loading and featurizing...")
X_tr, y_tr, X_val, y_val, y_std = load_data()
print(f"Train: {len(X_tr)}, Val: {len(X_val)}")

## 4. Train the MLP

Now let's train a simple MLP and see how well our features work!

In [ ]:
class MLP(nn.Module):
    @nn.compact
    def __call__(self, x):
        x  # [batch, feature_dim]
        x = nn.relu(nn.Dense(64)(x))  # [batch, 64]
        x = nn.relu(nn.Dense(32)(x))  # [batch, 32]
        return nn.Dense(1)(x).squeeze(-1)  # [batch]


@jax.jit
def step(params, opt_state, X, y):
    def loss_fn(p):
        return jnp.mean((MLP().apply(p, X) - y) ** 2)

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optax.adam(1e-3).update(grads, opt_state, params)
    return optax.apply_updates(params, updates), opt_state, loss

In [ ]:
n_epochs = 100
batch_size = 64

model = MLP()

# Initialize
params = model.init(jax.random.PRNGKey(0), X_tr[:1])
opt_state = optax.adam(1e-3).init(params)

# Initialize loss lists
train_rmses = []
val_rmses = []

# Train
for epoch in range(n_epochs):
    perm = np.random.permutation(len(X_tr))
    for i in range(0, len(X_tr), batch_size):
        idx = perm[i : i + batch_size]
        params, opt_state, _ = step(params, opt_state, X_tr[idx], y_tr[idx])

    if (epoch + 1) % 10 == 0:
        train_rmse = jnp.sqrt(jnp.mean((model.apply(params, X_tr) - y_tr) ** 2)) * y_std
        val_rmse = jnp.sqrt(jnp.mean((model.apply(params, X_val) - y_val) ** 2)) * y_std

        train_rmses.append(train_rmse)
        val_rmses.append(val_rmse)

        print(f"Epoch {epoch + 1:3d} | Train: {train_rmse:.3f} eV | Val: {val_rmse:.3f} eV")

In [ ]:
# Plot train and validation RMSE
plt.plot(np.arange(10, n_epochs + 1, 10), train_rmses, label="Train RMSE")
plt.plot(np.arange(10, n_epochs + 1, 10), val_rmses, label="Validation RMSE")
plt.xlabel("Epoch")
plt.ylabel("RMSE (eV)")
plt.title("Train and Validation RMSE over Epochs")
plt.legend()
plt.show()

In [ ]:
# Plot y_pred vs y_true
y_pred_val = model.apply(params, X_val) * y_std
y_true_val = y_val * y_std

plt.figure(figsize=(5, 5))
plt.scatter(y_true_val, y_pred_val, alpha=0.3, s=5)
plt.plot([y_true_val.min(), y_true_val.max()], [y_true_val.min(), y_true_val.max()], "k--", lw=1)
plt.xlabel("True HOMO-LUMO gap (eV)")
plt.ylabel("Predicted HOMO-LUMO gap (eV)")
plt.title(f"Validation set (RMSE: {val_rmses[-1]:.3f} eV)")
plt.axis("equal")
plt.tight_layout()
plt.show()